# Hackathon 2026 - Previsão Climática de Precipitação sobre a América do Sul

## Integrantes

* Bruno Antico Galin (@brunoantico)
* Lucas Pires de Camargo Sarai (@lsarai)
* Victor Luiz de Sá Alves (@viktoralves)


## Analisando dados disponibilizados

### Dados de Treino

In [1]:
%pip install tensorflow keras-tuner

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import xarray as xr
import pandas as pd
import kagglehub
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import numpy as np
import tensorflow as tf

def mostrar_infos_df(df: pd.DataFrame, label: str):
    print("#"*5,label.upper(),"#"*5)
    print(df.info())
    display(df.head())
    print("#"*20,"\n")

def contar_valores_por_time(df: pd.DataFrame) -> pd.DataFrame:
    if "time" not in df.index.names:
        raise ValueError("O DataFrame precisa ter `time` como índice.")

    valores = df.groupby(level="time").count()
    valores.index.name = "time"
    return valores

def contar_linhas_por_time(df: pd.DataFrame) -> pd.Series:
    return df.groupby(level="time").size().rename("total_registros")

def contar_dimensoes_grade(df: pd.DataFrame) -> dict[str, pd.Series]:
    nomes = set(df.index.names)
    esperados = {"time", "lat", "lon"}
    ausentes = esperados - nomes
    if ausentes:
        raise ValueError(f"O índice precisa conter: {sorted(esperados)}. Ausentes: {sorted(ausentes)}")

    coordenadas = df.index.to_frame(index=False)
    return {
        "latitudes_por_time": coordenadas.groupby("time")["lat"].nunique(),
        "longitudes_por_latitude": coordenadas.groupby("lat")["lon"].nunique(),
        "longitudes_por_time": coordenadas.groupby("time")["lon"].count(),
    }


In [3]:
path = kagglehub.competition_download('previsao-climatica-de-precipitacao-sobre-a-america-do-sul')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/previsao-climatica-de-precipitacao-sobre-a-america-do-sul


In [4]:
print("#"*5,"TREINO","#"*5,"\n")
tp = xr.open_dataset(f"{path}/treino_tp.nc").to_dataframe()              # (time, lat, lon), mm/dia


mostrar_infos_df(tp,"precipitação mes atual")
display(contar_linhas_por_time(tp))
display(contar_valores_por_time(tp))
display(contar_dimensoes_grade(tp))

print(tp.index.get_level_values("lat").unique())
print(tp.index.get_level_values("lon").unique())



##### TREINO ##### 



##### PRECIPITAÇÃO MES ATUAL #####


<class 'pandas.core.frame.DataFrame'>
MultiIndex: 78246756 entries, (Timestamp('1940-01-01 00:00:00'), np.float64(-60.0), np.float64(-90.0)) to (Timestamp('2022-12-01 00:00:00'), np.float64(15.0), np.float64(-25.0))
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   tp      float32
dtypes: float32(1)
memory usage: 746.3 MB
None


tp
time       lat   lon             
1940-01-01 -60.0 -90.00  2.254486
                 -89.75  2.285004
                 -89.50  2.338409
                 -89.25  2.424240
                 -89.00  2.477646

#################### 



time
1940-01-01    78561
1940-02-01    78561
1940-03-01    78561
1940-04-01    78561
1940-05-01    78561
              ...  
2022-08-01    78561
2022-09-01    78561
2022-10-01    78561
2022-11-01    78561
2022-12-01    78561
Name: total_registros, Length: 996, dtype: int64

,tp
time,
1940-01-01,78561
1940-02-01,78561
1940-03-01,78561
1940-04-01,78561
1940-05-01,78561
...,...
2022-08-01,78561
2022-09-01,78561
2022-10-01,78561


{'latitudes_por_time': time
 1940-01-01    301
 1940-02-01    301
 1940-03-01    301
 1940-04-01    301
 1940-05-01    301
              ... 
 2022-08-01    301
 2022-09-01    301
 2022-10-01    301
 2022-11-01    301
 2022-12-01    301
 Name: lat, Length: 996, dtype: int64,
 'longitudes_por_latitude': lat
 -60.00    261
 -59.75    261
 -59.50    261
 -59.25    261
 -59.00    261
          ... 
  14.00    261
  14.25    261
  14.50    261
  14.75    261
  15.00    261
 Name: lon, Length: 301, dtype: int64,
 'longitudes_por_time': time
 1940-01-01    78561
 1940-02-01    78561
 1940-03-01    78561
 1940-04-01    78561
 1940-05-01    78561
               ...  
 2022-08-01    78561
 2022-09-01    78561
 2022-10-01    78561
 2022-11-01    78561
 2022-12-01    78561
 Name: lon, Length: 996, dtype: int64}

Index([ -60.0, -59.75,  -59.5, -59.25,  -59.0, -58.75,  -58.5, -58.25,  -58.0,
       -57.75,
       ...
        12.75,   13.0,  13.25,   13.5,  13.75,   14.0,  14.25,   14.5,  14.75,
         15.0],
      dtype='float64', name='lat', length=301)


Index([ -90.0, -89.75,  -89.5, -89.25,  -89.0, -88.75,  -88.5, -88.25,  -88.0,
       -87.75,
       ...
       -27.25,  -27.0, -26.75,  -26.5, -26.25,  -26.0, -25.75,  -25.5, -25.25,
        -25.0],
      dtype='float64', name='lon', length=261)


In [5]:
alvo = xr.open_dataset(f"{path}/treino_tp_alvo.nc").to_dataframe()   # precipitação de M+1

mostrar_infos_df(alvo,"precipitação mes seguinte")
display(contar_linhas_por_time(alvo))
display(contar_valores_por_time(alvo))

##### PRECIPITAÇÃO MES SEGUINTE #####


<class 'pandas.core.frame.DataFrame'>
MultiIndex: 78246756 entries, (Timestamp('1940-01-01 00:00:00'), np.float64(-60.0), np.float64(-90.0)) to (Timestamp('2022-12-01 00:00:00'), np.float64(15.0), np.float64(-25.0))
Data columns (total 1 columns):
 #   Column   Dtype  
---  ------   -----  
 0   tp_alvo  float32
dtypes: float32(1)
memory usage: 746.3 MB
None


tp_alvo
time       lat   lon             
1940-01-01 -60.0 -90.00  3.153801
                 -89.75  3.170013
                 -89.50  3.149986
                 -89.25  3.075600
                 -89.00  2.992630

#################### 



time
1940-01-01    78561
1940-02-01    78561
1940-03-01    78561
1940-04-01    78561
1940-05-01    78561
              ...  
2022-08-01    78561
2022-09-01    78561
2022-10-01    78561
2022-11-01    78561
2022-12-01    78561
Name: total_registros, Length: 996, dtype: int64

,tp_alvo
time,
1940-01-01,78561
1940-02-01,78561
1940-03-01,78561
1940-04-01,78561
1940-05-01,78561
...,...
2022-08-01,78561
2022-09-01,78561
2022-10-01,78561


In [6]:
cobertura_nuvem = xr.open_dataset(f"{path}/treino_cloud_cover.nc").to_dataframe()

mostrar_infos_df(cobertura_nuvem,"cobertura nuvem")
display(contar_valores_por_time(cobertura_nuvem))
display(contar_linhas_por_time(cobertura_nuvem))

##### COBERTURA NUVEM #####


<class 'pandas.core.frame.DataFrame'>
MultiIndex: 78246756 entries, (Timestamp('1940-01-01 00:00:00'), np.float64(-60.0), np.float64(-90.0)) to (Timestamp('2022-12-01 00:00:00'), np.float64(15.0), np.float64(-25.0))
Data columns (total 1 columns):
 #   Column       Dtype  
---  ------       -----  
 0   cloud_cover  float32
dtypes: float32(1)
memory usage: 746.3 MB
None


cloud_cover
time       lat   lon                
1940-01-01 -60.0 -90.00     0.780718
                 -89.75     0.783556
                 -89.50     0.784548
                 -89.25     0.782808
                 -89.00     0.781160

#################### 



,cloud_cover
time,
1940-01-01,78561
1940-02-01,78561
1940-03-01,78561
1940-04-01,78561
1940-05-01,78561
...,...
2022-08-01,78561
2022-09-01,78561
2022-10-01,78561


time
1940-01-01    78561
1940-02-01    78561
1940-03-01    78561
1940-04-01    78561
1940-05-01    78561
              ...  
2022-08-01    78561
2022-09-01    78561
2022-10-01    78561
2022-11-01    78561
2022-12-01    78561
Name: total_registros, Length: 996, dtype: int64

In [7]:
pressao_superficie = xr.open_dataset(f"{path}/treino_surface_pressure.nc").to_dataframe()
geopotencial = xr.open_dataset(f"{path}/treino_geopotential_850.nc").to_dataframe()
umidade = xr.open_dataset(f"{path}/treino_shum_850.nc").to_dataframe()
umidade_relativa = xr.open_dataset(f"{path}/treino_rel_hum_850.nc").to_dataframe()
vento_u = xr.open_dataset(f"{path}/treino_u_850.nc").to_dataframe()
vento_v = xr.open_dataset(f"{path}/treino_v_850.nc").to_dataframe()
temperatura = xr.open_dataset(f"{path}/treino_temperature_850.nc").to_dataframe()
temp_2_metros = xr.open_dataset(f"{path}/treino_t2.nc").to_dataframe()

### Resumo

Rodamos com todas as variáveis e todas seguem o mesmo padrão, então com as de cima simplificamos e só importamos os dados

As conclusões foram que seguem exatamente a descrição da competicão:
1. Existem 996 meses em análise (jan/1940 a dez/2022)
2. Para cada mês, existem latitudes de 60.0°S a 15.0°N, com os valores avançando 0.25 (ex: 60.0,59.75,59.5,59.25,...), então são 60+15=75 valores inteiros com 4 subdivisões para cada, o que resulta em 75*4 = 300, só que +1 contando 60.0°S, então **301 latitudes**
3. Para cada latitudade, existem longitudes de 90°O a 25°O, com os valores avançando 0.25, então são 90-25=65 valores inteiros com 4 subdivisões cada, o que resulta em 65*4 + 1 = **261 longitudes por latitude**
4. Para cada mês, portanto, teremos **301x261 = 78.561 coordenadas por mês**, represenbtando os **78.561 pontos de grade** descritos na competição
5. Para cada mês, em cada um desses **78.561 pontos de grade**, teremos um valor associado, que varia com a tabela (ex: pode ser temperatura, pressão, umidade relativa, etc.)
6. O dataset **`tp`** possui os valores reais dos pontos na grade daquele mês, enquanto **`tp_alvo`** possui um deslocamento de 1 mês, então cada ponto está com o valor daquele mesmo ponto no mês seguinte (ex: o valor de **`tp_alvo`**(jan/1940, lat: 60°S, lon: 90°O) = 3.153801, equivale ao valor de **`tp`**(fev/1940, lat: 60°S, lon: 90°O)=3.153801). Dessa forma, o último mês do dataset, dez/2022, possui os valores reais da grade em **`tp`**, mas todos os valores da grade estão como `NaN` em **`tp_alvo`**, pois não há mês seguinte.
 


### Teste

In [8]:
print("#"*5,"TESTE","#"*5,"\n")

teste = xr.open_dataset(f"{path}/teste_features.nc").to_dataframe()

mostrar_infos_df(teste,"teste")
display(contar_linhas_por_time(teste))
display(contar_valores_por_time(teste))
display(contar_dimensoes_grade(teste))


##### TESTE ##### 



##### TESTE #####
<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1885464 entries, (Timestamp('2023-01-01 00:00:00'), np.float64(-60.0), np.float64(-90.0)) to (Timestamp('2024-12-01 00:00:00'), np.float64(15.0), np.float64(-25.0))
Data columns (total 13 columns):
 #   Column            Dtype         
---  ------            -----         
 0   t2                float32       
 1   cloud_cover       float32       
 2   shum_850          float32       
 3   surface_pressure  float32       
 4   u_850             float32       
 5   v_850             float32       
 6   temperature_850   float32       
 7   rel_hum_850       float32       
 8   geopotential_850  float32       
 9   tp_alvo           float32       
 10  tp_ultima_obs     float32       
 11  time_origem       datetime64[ns]
 12  lag_meses         int16         
dtypes: datetime64[ns](1), float32(11), int16(1)
memory usage: 106.1 MB
None


t2  cloud_cover  shum_850  surface_pressure  \
time       lat   lon                                                           
2023-01-01 -60.0 -90.00  276.325195     0.756066  0.002307         98572.875   
                 -89.75  276.309570     0.754373  0.002310         98577.875   
                 -89.50  276.297852     0.752557  0.002313         98576.875   
                 -89.25  276.286133     0.750558  0.002316         98567.875   
                 -89.00  276.284180     0.749459  0.002321         98563.875   

                             u_850     v_850  temperature_850  rel_hum_850  \
time       lat   lon                                                         
2023-01-01 -60.0 -90.00  10.837509 -1.043722       267.944336    77.139938   
                 -89.75  10.828720 -1.022238       267.959961    77.075485   
                 -89.50  10.815048 -1.002218       267.975586    76.973923   
                 -89.25  10.800400 -0.985128       267.992188    76.891891   
                 -89.00  10.782822 -0.970968       268.010742    76.891891   

                         geopotential_850  tp_alvo  tp_ultima_obs time_origem  \
time       lat   lon                                                            
2023-01-01 -60.0 -90.00      11558.367188      NaN       1.911163  2022-12-01   
                 -89.75      11560.492188      NaN       1.895905  2022-12-01   
                 -89.50      11562.617188      NaN       1.913071  2022-12-01   
                 -89.25      11564.867188      NaN       1.974106  2022-12-01   
                 -89.00      11566.992188      NaN       2.016068  2022-12-01   

                         lag_meses  
time       lat   lon                
2023-01-01 -60.0 -90.00          1  
                 -89.75          1  
                 -89.50          1  
                 -89.25          1  
                 -89.00          1

#################### 



time
2023-01-01    78561
2023-02-01    78561
2023-03-01    78561
2023-04-01    78561
2023-05-01    78561
2023-06-01    78561
2023-07-01    78561
2023-08-01    78561
2023-09-01    78561
2023-10-01    78561
2023-11-01    78561
2023-12-01    78561
2024-01-01    78561
2024-02-01    78561
2024-03-01    78561
2024-04-01    78561
2024-05-01    78561
2024-06-01    78561
2024-07-01    78561
2024-08-01    78561
2024-09-01    78561
2024-10-01    78561
2024-11-01    78561
2024-12-01    78561
Name: total_registros, dtype: int64

,t2,cloud_cover,shum_850,surface_pressure,u_850,v_850,temperature_850,rel_hum_850,geopotential_850,tp_alvo,tp_ultima_obs,time_origem,lag_meses
time,,,,,,,,,,,,,
2023-01-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561
2023-02-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561
2023-03-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561
2023-04-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561
2023-05-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561
2023-06-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561
2023-07-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561
2023-08-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561
2023-09-01,78561,78561,78561,78561,78561,78561,78561,78561,78561,0,78561,78561,78561


{'latitudes_por_time': time
 2023-01-01    301
 2023-02-01    301
 2023-03-01    301
 2023-04-01    301
 2023-05-01    301
 2023-06-01    301
 2023-07-01    301
 2023-08-01    301
 2023-09-01    301
 2023-10-01    301
 2023-11-01    301
 2023-12-01    301
 2024-01-01    301
 2024-02-01    301
 2024-03-01    301
 2024-04-01    301
 2024-05-01    301
 2024-06-01    301
 2024-07-01    301
 2024-08-01    301
 2024-09-01    301
 2024-10-01    301
 2024-11-01    301
 2024-12-01    301
 Name: lat, dtype: int64,
 'longitudes_por_latitude': lat
 -60.00    261
 -59.75    261
 -59.50    261
 -59.25    261
 -59.00    261
          ... 
  14.00    261
  14.25    261
  14.50    261
  14.75    261
  15.00    261
 Name: lon, Length: 301, dtype: int64,
 'longitudes_por_time': time
 2023-01-01    78561
 2023-02-01    78561
 2023-03-01    78561
 2023-04-01    78561
 2023-05-01    78561
 2023-06-01    78561
 2023-07-01    78561
 2023-08-01    78561
 2023-09-01    78561
 2023-10-01    78561
 2023-11-01    

### Resumo

Para a tabela de teste, temos uma modificação.

Ao invés de separar em tabelas com cada variável, elas foram unidas em uma única tabela. Dessa forma, cada registro representa um ponto na grade para um determinado mês
 


## ConvLSTM para previsão de precipitação

Cada mês é tratado como um frame `lat x lon`, cada variável climática como um canal e o alvo como a precipitação do mês seguinte.

In [ ]:
"""ConvLSTM para previsão mensal de precipitação na grade lat/lon."""

BASE_FEATURE_FILES = {
    "tp": "treino_tp.nc",
    "cloud_cover": "treino_cloud_cover.nc",
    "surface_pressure": "treino_surface_pressure.nc",
    "geopotential_850": "treino_geopotential_850.nc",
    "shum_850": "treino_shum_850.nc",
    "rel_hum_850": "treino_rel_hum_850.nc",
    "u_850": "treino_u_850.nc",
    "v_850": "treino_v_850.nc",
    "temperature_850": "treino_temperature_850.nc",
    "t2": "treino_t2.nc",
}

INPUT_FEATURE_ORDER = [
    "cloud_cover",
    "geopotential_850",
    "surface_pressure",
    "rel_hum_850",
    "shum_850",
    "u_850",
    "v_850",
    "temperature_850",
    "t2",
]

FEATURE_FILES = BASE_FEATURE_FILES


def selecionar_device():
    """Seleciona TPU, depois GPU e, por último, CPU para o TensorFlow."""
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        return resolver.master()
    except (ValueError, RuntimeError):
        if tf.config.list_physical_devices("GPU"):
            return "/GPU:0"
        return "/CPU:0"


def abrir_datasets(data_dir: str | Path):
    """Abre os canais de entrada e o alvo sem materializar os dados na memória."""
    data_dir = Path(data_dir)
    features = {}
    for name, filename in FEATURE_FILES.items():
        dataset = xr.open_dataset(data_dir / filename)
        variable = next(iter(dataset.data_vars))
        features[name] = (dataset, variable)

    target = xr.open_dataset(data_dir / "treino_tp_alvo.nc")
    target_variable = str(next(iter(target.data_vars)))
    return features, target, target_variable


def extrair_feature(feature_name: str, feature_datasets, time_indices):
    """Obtém um canal bruto para um intervalo de tempo."""
    if feature_name in INPUT_FEATURE_ORDER:
        dataset, variable = feature_datasets[feature_name]
        return (
            dataset[variable]
            .isel(time=time_indices)
            .transpose("time", "lat", "lon")
            .values
        )

    raise ValueError(f"Canal não suportado: {feature_name}")


class ClimateGridDataset:
    """Monta uma sequência [tempo, lat, lon, canais] sob demanda."""

    def __init__(
        self,
        ends: Iterable[int],
        sequence_length: int,
        feature_datasets,
        target_dataset,
        target_variable: str,
        feature_names: Iterable[str],
        means: dict[str, float],
        stds: dict[str, float],
        target_mean: float,
        target_std: float,
    ):
        self.ends = np.asarray(list(ends), dtype=np.int64)
        self.sequence_length = sequence_length
        self.feature_datasets = feature_datasets
        self.target_dataset = target_dataset
        self.target_variable = target_variable
        self.feature_names = list(feature_names)
        self.means = means
        self.stds = stds
        self.target_mean = target_mean
        self.target_std = target_std

    def __len__(self):
        return len(self.ends)

    def __getitem__(self, index):
        end = int(self.ends[index])
        indices = np.arange(end - self.sequence_length + 1, end + 1)
        channels = []

        for name in self.feature_names:
            frame = np.asarray(extrair_feature(name, self.feature_datasets, indices), dtype=np.float32)
            frame = (frame - self.means[name]) / self.stds[name]
            channels.append(np.nan_to_num(frame, nan=0.0))

        x = np.stack(channels, axis=1)
        y = (
            self.target_dataset[self.target_variable]
            .isel(time=end)
            .transpose("lat", "lon")
            .values
        )
        y = (np.asarray(y, dtype=np.float32) - self.target_mean) / self.target_std
        y = np.nan_to_num(y, nan=0.0)
        x = np.transpose(x, (0, 2, 3, 1))
        return x.astype(np.float32), y[..., np.newaxis].astype(np.float32)


def _criar_tf_dataset(dataset, batch_size, spatial_shape, shuffle=False):
    n_lat, n_lon = spatial_shape
    output_signature = (
        tf.TensorSpec(
            shape=(dataset.sequence_length, n_lat, n_lon, len(dataset.feature_names)),
            dtype=tf.float32,
        ),
        tf.TensorSpec(shape=(n_lat, n_lon, 1), dtype=tf.float32),
    )
    result = tf.data.Dataset.from_generator(
        lambda: (dataset[index] for index in range(len(dataset))),
        output_signature=output_signature,
    )
    if shuffle:
        result = result.shuffle(
            min(max(len(dataset), 1), 128), reshuffle_each_iteration=True
        )
    result = result.batch(batch_size)
    batch_count = (len(dataset) + batch_size - 1) // batch_size
    result = result.apply(tf.data.experimental.assert_cardinality(batch_count))
    return result.prefetch(tf.data.AUTOTUNE)


def preparar_dados(
    data_dir: str | Path,
    sequence_length: int = 3,
    batch_size: int = 2,
    train_fraction: float = 0.8,
    validation_fraction: float = 0.1,
    num_workers: int = 2,
):
    """Cria loaders cronológicos e normaliza usando somente o treino.

    A entrada usa nove canais físicos brutos, sem features derivadas.
    """
    features, target, target_variable = abrir_datasets(data_dir)
    times = pd.DatetimeIndex(features["tp"][0].time.values)
    n_times = len(times)
    train_end = int(n_times * train_fraction)
    validation_end = int(n_times * (train_fraction + validation_fraction))

    valid_ends = np.arange(sequence_length - 1, n_times)
    train_ends = valid_ends[valid_ends < train_end]
    validation_ends = valid_ends[
        (valid_ends >= train_end) & (valid_ends < validation_end)
    ]
    test_ends = valid_ends[valid_ends >= validation_end]

    train_indices = np.arange(train_end)
    means, stds = {}, {}
    for name in INPUT_FEATURE_ORDER:
        values = np.asarray(
            extrair_feature(name, features, train_indices),
            dtype=np.float32,
        )
        means[name] = float(np.nanmean(values))
        stds[name] = max(float(np.nanstd(values)), 1e-6)

    target_values = target[target_variable].isel(time=train_indices).values
    target_mean = float(np.nanmean(target_values))
    target_std = max(float(np.nanstd(target_values)), 1e-6)
    spatial_shape = (
        features["tp"][0].sizes["lat"],
        features["tp"][0].sizes["lon"],
    )
    args = (
        features,
        target,
        target_variable,
        INPUT_FEATURE_ORDER,
        means,
        stds,
        target_mean,
        target_std,
    )

    datasets = {
        "train": ClimateGridDataset(train_ends, sequence_length, *args),
        "validation": ClimateGridDataset(validation_ends, sequence_length, *args),
        "test": ClimateGridDataset(test_ends, sequence_length, *args),
    }
    if not len(datasets["train"]):
        raise ValueError("A divisão de treino não contém amostras.")
    if not len(datasets["validation"]):
        raise ValueError("A divisão de validação não contém amostras.")
    loaders = {
        "train": _criar_tf_dataset(
            datasets["train"], batch_size, spatial_shape, shuffle=True
        ),
        "validation": _criar_tf_dataset(
            datasets["validation"], batch_size, spatial_shape
        ),
        "test": _criar_tf_dataset(datasets["test"], batch_size, spatial_shape),
    }
    metadata = {
        "times": times,
        "n_channels": len(INPUT_FEATURE_ORDER),
        "n_lat": features["tp"][0].sizes["lat"],
        "n_lon": features["tp"][0].sizes["lon"],
        "target_mean": target_mean,
        "target_std": target_std,
        "feature_means": means,
        "feature_stds": stds,
    }
    return loaders, metadata


class ConvLSTMPrecipitacao(tf.keras.Model):
    """ConvLSTM baseada na camada oficial ConvLSTM2D do Keras."""

    def __init__(
        self,
        input_channels: int = 6,
        hidden_channels: int = 32,
        kernel_size: int = 3,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.input_channels = input_channels
        self.hidden_channels = hidden_channels
        self.conv_lstm = tf.keras.layers.ConvLSTM2D(
            filters=hidden_channels,
            kernel_size=(kernel_size, kernel_size),
            padding="same",
            return_sequences=False,
            dropout=dropout,
        )
        self.head = tf.keras.layers.Conv2D(1, kernel_size=1)

    def build(self, input_shape):
        input_shape = tf.TensorShape(input_shape)
        if input_shape.rank != 5:
            raise ValueError(
                "ConvLSTMPrecipitacao espera entradas com forma "
                "(batch, tempo, lat, lon, canais)."
            )
        if any(input_shape[index] is None for index in (2, 3, 4)):
            raise ValueError(
                "As dimensoes tempo, lat, lon e canais precisam ser estaticas "
                "para construir a ConvLSTM."
            )

        self.conv_lstm.build(input_shape)
        conv_output_shape = self.conv_lstm.compute_output_shape(input_shape)
        self.head.build(conv_output_shape)
        super().build(input_shape)

    def call(self, inputs, training=None, mask=None):
        return self.head(self.conv_lstm(inputs, training=training))


def _compilar_modelo(model, learning_rate=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.MeanSquaredError(),
    )
    return model


def avaliar(model, loader, device, loss_function):
    del device, loss_function
    return float(model.evaluate(loader, verbose=0, return_dict=False))


def treinar(
    model,
    train_loader,
    validation_loader,
    device,
    epochs=10,
    lr=1e-3,
    xla_module=None,
):
    del xla_module
    with tf.device(device):
        if model.optimizer is None:
            _compilar_modelo(model, lr)
        history = model.fit(
            train_loader,
            validation_data=validation_loader,
            epochs=epochs,
            verbose=1,
        ).history
    return {"train": history["loss"], "validation": history["val_loss"]}


def otimizar_hyperparametros(
    train_loader,
    validation_loader,
    input_shape,
    max_trials=8,
    epochs=10,
    diretorio="keras_tuner",
):
    """Busca a configuração ConvLSTM com Keras Tuner e retorna o melhor modelo."""
    try:
        import keras_tuner as kt
    except ImportError as error:
        raise ImportError(
            "Instale Keras Tuner com `pip install keras-tuner`."
        ) from error

    def construir(hp):
        model = ConvLSTMPrecipitacao(
            input_channels=input_shape[-1],
            hidden_channels=hp.Int("hidden_channels", 16, 64, step=16),
            kernel_size=hp.Choice("kernel_size", [3, 5]),
            dropout=hp.Float("dropout", 0.0, 0.3, step=0.1),
        )
        model.build((None, *input_shape))
        return _compilar_modelo(
            model,
            hp.Float("learning_rate", 1e-4, 3e-3, sampling="log"),
        )

    tuner = kt.RandomSearch(
        construir,
        objective="val_loss",
        max_trials=max_trials,
        directory=diretorio,
        project_name="conv_lstm_precipitacao",
        overwrite=True,
    )
    tuner.search(train_loader, validation_data=validation_loader, epochs=epochs, verbose=1)
    melhor_modelo = tuner.get_best_models(num_models=1)[0]
    return melhor_modelo, tuner


def prever(model, loader, device, target_mean, target_std):
    """Gera previsões na escala original do alvo, revertendo a padronização."""
    del device
    predictions = model.predict(loader, verbose=0)
    return predictions * target_std + target_mean


def calcular_rmse(model, loader, device, target_mean, target_std):
    """Calcula RMSE na escala original do alvo para um loader rotulado."""
    soma_erro_quadratico = 0.0
    quantidade_valores = 0
    del device

    for x, y in loader:
        previsao = model(x, training=False).numpy()
        real = y.numpy()
        erro = (previsao - real) * target_std
        soma_erro_quadratico += float(np.sum(erro**2))
        quantidade_valores += erro.size

    if quantidade_valores == 0:
        raise ValueError("O loader não contém amostras para avaliação.")

    return float(np.sqrt(soma_erro_quadratico / quantidade_valores))


def extrair_feature_teste(frame, feature_name: str, latitudes, longitudes):
    """Extrai um canal físico bruto do dataframe de teste."""
    if feature_name in INPUT_FEATURE_ORDER:
        grid = frame[feature_name].unstack("lon").reindex(
            index=latitudes, columns=longitudes
        )
        return grid.to_numpy(dtype=np.float32)

    raise ValueError(f"Canal de teste não suportado: {feature_name}")


def previsoes_teste(
    model,
    teste_df,
    device,
    feature_means,
    feature_stds,
    target_mean,
    target_std,
    batch_size=2,
):
    """Gera previsões para os meses do arquivo oficial de teste."""
    required = {
        "cloud_cover",
        "geopotential_850",
        "surface_pressure",
        "rel_hum_850",
        "shum_850",
        "u_850",
        "v_850",
        "temperature_850",
        "t2",
        "tp_ultima_obs",
    }
    missing = required - set(teste_df.columns)
    if missing:
        raise ValueError(f"Colunas ausentes no teste: {sorted(missing)}")

    times = pd.DatetimeIndex(
        teste_df.index.get_level_values("time").unique()
    ).sort_values()
    times = times[
        (times >= pd.Timestamp("2023-01-01"))
        & (times <= pd.Timestamp("2024-12-01"))
    ]
    latitudes = np.sort(teste_df.index.get_level_values("lat").unique())
    longitudes = np.sort(teste_df.index.get_level_values("lon").unique())
    inputs = []

    for time in times:
        frame = teste_df.loc[time].reset_index().set_index(["lat", "lon"])
        channels = []
        for name in INPUT_FEATURE_ORDER:
            grid = extrair_feature_teste(frame, name, latitudes, longitudes)
            grid = (grid - feature_means[name]) / feature_stds[name]
            channels.append(np.nan_to_num(grid, nan=0.0))

        grid = np.stack(channels, axis=-1)
        inputs.append(grid[np.newaxis, ...])

    del device
    predicted_grids = []
    for start in range(0, len(inputs), batch_size):
        x = np.concatenate(inputs[start : start + batch_size], axis=0)
        predicted_grids.append(
            model.predict(x, verbose=0)[:, :, :, 0] * target_std + target_mean
        )

    return np.concatenate(predicted_grids)


def avaliar_teste_contra_ultima_obs(
    model,
    teste_df,
    device,
    feature_means,
    feature_stds,
    target_mean,
    target_std,
    batch_size=2,
):
    """Prevê tp no teste e compara com `tp_ultima_obs` por mês.

    Esta é uma comparação de referência/persistência. `tp_ultima_obs` é o
    último valor observado, não o valor real do mês previsto.
    """
    required = {
        "cloud_cover",
        "geopotential_850",
        "surface_pressure",
        "rel_hum_850",
        "shum_850",
        "u_850",
        "v_850",
        "temperature_850",
        "t2",
        "tp_ultima_obs",
    }
    missing = required - set(teste_df.columns)
    if missing:
        raise ValueError(f"Colunas ausentes no teste: {sorted(missing)}")

    times = pd.DatetimeIndex(teste_df.index.get_level_values("time").unique()).sort_values()
    times = times[
        (times >= pd.Timestamp("2023-01-01"))
        & (times <= pd.Timestamp("2024-12-01"))
    ]
    latitudes = np.sort(teste_df.index.get_level_values("lat").unique())
    longitudes = np.sort(teste_df.index.get_level_values("lon").unique())
    predictions, references, dates = [], [], []

    for time in times:
        frame = teste_df.loc[time].reset_index().set_index(["lat", "lon"])
        channels = []
        for name in INPUT_FEATURE_ORDER:
            grid = extrair_feature_teste(frame, name, latitudes, longitudes)
            grid = (grid - feature_means[name]) / feature_stds[name]
            channels.append(np.nan_to_num(grid, nan=0.0))

        grid = np.stack(channels, axis=-1)
        x = grid[np.newaxis, np.newaxis, ...]
        reference = frame["tp_ultima_obs"].unstack("lon").reindex(
            index=latitudes, columns=longitudes
        ).to_numpy(dtype=np.float32)
        predictions.append(x)
        references.append(reference)
        dates.append(time)

    del device
    predicted_grids = []
    for start in range(0, len(predictions), batch_size):
        x = np.concatenate(predictions[start : start + batch_size], axis=0)
        predicted_grids.append(
            model.predict(x, verbose=0)[:, :, :, 0] * target_std + target_mean
        )

    predicted_grids = np.concatenate(predicted_grids)
    references = np.asarray(references)
    errors = predicted_grids - references
    monthly_rmse = np.sqrt(np.mean(errors**2, axis=(1, 2)))
    result = pd.DataFrame(
        {"time": dates, "rmse_tp_vs_tp_ultima_obs": monthly_rmse}
    ).set_index("time")
    global_rmse = float(np.sqrt(np.mean(errors**2)))
    return global_rmse, result, predicted_grids


def plotar_historico(history):
    """Plota a perda de treino e validação ao longo das épocas."""
    import matplotlib.pyplot as plt

    epocas = np.arange(1, len(history["train"]) + 1)
    plt.figure(figsize=(9, 5))
    plt.plot(epocas, history["train"], marker="o", label="Treinamento")
    plt.plot(epocas, history["validation"], marker="o", label="Validação")
    plt.xlabel("Época")
    plt.ylabel("MSE normalizado")
    plt.title("Perda durante o treinamento")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


def diagnosticar_device(model, device):
    """Confirma se modelo, CUDA e um batch estão no dispositivo esperado."""
    print("device selecionado:", device)
    print("dispositivos TensorFlow:", tf.config.list_logical_devices())
    print("modelo compilado:", model.optimizer is not None)


In [10]:
device = selecionar_device()
# O mês t prevê o mês t+1; por isso cada amostra usa apenas um mês de entrada.
loaders, convlstm_metadata = preparar_dados(path, sequence_length=1, batch_size=2, num_workers=2)
input_shape = (
    1,
    convlstm_metadata["n_lat"],
    convlstm_metadata["n_lon"],
    convlstm_metadata["n_channels"],
)
model, tuner = otimizar_hyperparametros(
    loaders["train"],
    loaders["validation"],
    input_shape=input_shape,
    max_trials=8,
    epochs=10,
)
print("Melhores hiperparâmetros:", tuner.get_best_hyperparameters(1)[0].values)
print(convlstm_metadata)
diagnosticar_device(model, device)

Trial 3 Complete [03h 51m 54s]
val_loss: 0.46605992317199707

Best val_loss So Far: 0.42012420296669006
Total elapsed time: 11h 34m 38s

Search: Running Trial #4

Value             |Best Value So Far |Hyperparameter
64                |64                |hidden_channels
3                 |5                 |kernel_size
0                 |0.1               |dropout
0.0003773         |0.0010512         |learning_rate



Epoch 1/10



  1/398 ━━━━━━━━━━━━━━━━━━━━ 22:44:24 206s/step - loss: 1.1212


  2/398 ━━━━━━━━━━━━━━━━━━━━ 1:57 297ms/step - loss: 1.1034   


  3/398 ━━━━━━━━━━━━━━━━━━━━ 11:02 2s/step - loss: 1.0933  

...


238/398 ━━━━━━━━━━━━━━━━━━━━ 8:12 3s/step - loss: 0.6799


239/398 ━━━━━━━━━━━━━━━━━━━━ 8:09 3s/step - loss: 0.6795


240/398 ━━━━━━━━━━━━━━━━━━━━ 8:06 3s/step - loss: 0.6791


241/398 ━━━━━━━━━━━━━━━━━━━━ 8:03 3s/step - loss: 0.6787


242/398 ━━━━━━━━━━━━━━━━━━━━ 8:00 3s/step - loss: 0.6784

### OBSERVAÇÃO

O kaggle não rodou as 8 iterações, mas apenas 4. 

Porém, para envio do `submission.csv` coletamos os arquivos `trial.json` e `computed.weights.h5` gerados pelo Keras Turner, fizemos um ajuste temporário nas células abaixo para o modelo ser carregado com esses parâmetros e rodamos até o final. 

Sendo assim, considere as células abaixo como o fluxo previsto, mas o motivo de não ter output explícito se deve pelo que foi posto acima.

In [ ]:
melhor_trial = tuner.oracle.get_best_trials(num_trials=1)[0]

historico = {
    "train": [
        observacao.value[0]
        for observacao in melhor_trial.metrics.get_history("loss")
    ],
    "validation": [
        observacao.value[0]
        for observacao in melhor_trial.metrics.get_history("val_loss")
    ],
}

plotar_historico(historico)

In [ ]:
# Previsões para os meses do conjunto de teste, após o treinamento.
previsoes = prever(
    model, loaders["test"], device,
    convlstm_metadata["target_mean"],
    convlstm_metadata["target_std"],
)
print(previsoes.shape)  # [amostras, 1, latitude, longitude]


### RMSE no conjunto de teste interno (10% de 1940 a 2022)

Este `test_loader` é a parte final do período de treino, que possui `tp_alvo` (10% do conjunto dedicado a teste)

In [ ]:
rmse_teste = calcular_rmse(
    model,
    loaders["test"],
    device,
    convlstm_metadata["target_mean"],
    convlstm_metadata["target_std"],
)
print(f"RMSE no teste: {rmse_teste:.6f} mm/dia")

## RMSE no conjunto oficial de teste (2023/2024)

Esta comparação usa `tp_ultima_obs` como referência de persistência da base de teste.

In [ ]:
rmse_tp_ultima_obs, rmse_mensal, previsoes_teste = avaliar_teste_contra_ultima_obs(
    model,
    teste,
    device,
    convlstm_metadata["feature_means"],
    convlstm_metadata["feature_stds"],
    convlstm_metadata["target_mean"],
    convlstm_metadata["target_std"],
)
print(f"RMSE global de tp contra tp_ultima_obs: {rmse_tp_ultima_obs:.6f} mm/dia")
display(rmse_mensal)
print("Formato das previsões:", previsoes_teste.shape)


### Gerando o arquivo de submissão para 2023–2024

As previsões são gravadas no mesmo formato de `sample_submission.csv`: uma linha por mês e ponto da grade.

In [ ]:
df_submission = pd.read_csv(f"{path}/sample_submission.csv")

datas_submissao = pd.DatetimeIndex(
    teste.index.get_level_values("time").unique()
).sort_values()
datas_submissao = datas_submissao[
    (datas_submissao >= pd.Timestamp("2023-01-01"))
    & (datas_submissao <= pd.Timestamp("2024-12-01"))
]
latitudes_submissao = np.sort(teste.index.get_level_values("lat").unique())
longitudes_submissao = np.sort(teste.index.get_level_values("lon").unique())

previsoes_teste = np.asarray(previsoes_teste)
esperado = (
    len(datas_submissao),
    len(latitudes_submissao),
    len(longitudes_submissao),
)
if previsoes_teste.shape != esperado:
    raise ValueError(
        f"Formato inesperado das previsões: {previsoes_teste.shape}; "
        f"esperado: {esperado}"
    )

ids_esperados = [
    f"{data:%Y_%m}_{latitude:.2f}_{longitude:.2f}"
    for data in datas_submissao
    for latitude in latitudes_submissao
    for longitude in longitudes_submissao
]
if ids_esperados != df_submission["id"].tolist():
    raise ValueError("A ordem das coordenadas não coincide com sample_submission.csv")

# O flatten preserva a ordem mês, latitude e longitude usada nos IDs.
df_submission_convlstm = df_submission.copy()
df_submission_convlstm["tp_mm_day"] = previsoes_teste.reshape(-1)

if len(df_submission_convlstm) != 1_885_464:
    raise ValueError(
        f"Quantidade inesperada de previsões: {len(df_submission_convlstm)}"
    )

arquivo_submissao = "submission.csv"
df_submission_convlstm.to_csv(arquivo_submissao, index=False)
print(f"Arquivo salvo: {arquivo_submissao}")
print(f"Linhas: {len(df_submission_convlstm):,}")
display(df_submission_convlstm.head())
display(df_submission_convlstm.tail())